<span style = "font-family: Verdana; font-size: 20px">

### **DATA UNDERSTANDING: Learning Behavior Analytics**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")

data_path = Path('../data/raw')

In [ ]:
df_assessments = pd.read_csv(data_path / 'assessments.csv')
df_courses = pd.read_csv(data_path / 'courses.csv')
df_student_assessment = pd.read_csv(data_path / 'studentAssessment.csv')
df_student_info = pd.read_csv(data_path / 'studentInfo.csv')
df_student_registration = pd.read_csv(data_path / 'studentRegistration.csv')
df_student_vle = pd.read_csv(data_path / 'studentVle.csv')
df_vle = pd.read_csv(data_path / 'vle.csv')

print(f"\nDatasets loaded:")
print(f"  - assessments.csv: {df_assessments.shape[0]} rows x {df_assessments.shape[1]} columns")
print(f"  - courses.csv: {df_courses.shape[0]} rows x {df_courses.shape[1]} columns")
print(f"  - studentAssessment.csv: {df_student_assessment.shape[0]} rows x {df_student_assessment.shape[1]} columns")
print(f"  - studentInfo.csv: {df_student_info.shape[0]} rows x {df_student_info.shape[1]} columns")
print(f"  - studentRegistration.csv: {df_student_registration.shape[0]} rows x {df_student_registration.shape[1]} columns")
print(f"  - studentVle.csv: {df_student_vle.shape[0]} rows x {df_student_vle.shape[1]} columns")
print(f"  - vle.csv: {df_vle.shape[0]} rows x {df_vle.shape[1]} columns")


Datasets loaded:
  - courses.csv: 22 rows x 3 columns
  - studentInfo.csv: 32593 rows x 12 columns
  - studentRegistration.csv: 32593 rows x 5 columns
  - studentVle.csv: 10655280 rows x 6 columns
  - assessments.csv: 206 rows x 6 columns
  - studentAssessment.csv: 173912 rows x 5 columns
  - vle.csv: 6364 rows x 6 columns


<span style = "font-family: Verdana; font-size: 15px">

### courses.csv
A directory of all classes offered.

- `code_module`: The subject name - **primary key**.

- `code_presentation`: When it started (e.g., B for Feb, J for Oct) - **primary key**.

- `length`: How many days the course lasted.

In [3]:
print(f"Shape: {df_courses.shape[0]} rows × {df_courses.shape[1]} columns\n")
print("Columns:")
print(df_courses.dtypes)
print("\nFirst 5 rows:")
print(df_courses.head())
print("\nDataset Info:")
print(f"Memory usage: {df_courses.memory_usage(deep=True).sum() / 1024:.2f} KB")
print("\nMissing values:")
print(df_courses.isnull().sum())
print("\nUnique modules:")
print(f"Number of unique modules: {df_courses['code_module'].nunique()}")

Shape: 22 rows × 3 columns

Columns:
code_module                   object
code_presentation             object
module_presentation_length     int64
dtype: object

First 5 rows:
  code_module code_presentation  module_presentation_length
0         AAA             2013J                         268
1         AAA             2014J                         269
2         BBB             2013J                         268
3         BBB             2014J                         262
4         BBB             2013B                         240

Dataset Info:
Memory usage: 2.58 KB

Missing values:
code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64

Unique modules:
Number of unique modules: 7


<span style = "font-family: Verdana; font-size: 15px">

### assessment.csv
The "Syllabus" or grading plan for each course.

- `code_module` - **foreign key** from *courses.csv*.

- `code_presentation` - **foreign key** from *courses.csv*.

- `id_assessment`: identification number of the assessment - **primary key**.

- `assessment_type`: type of assessment. Three types of assessments exist: Tutor Marked Assessment (TMA), Computer Marked Assessment (CMA) and Final Exam (Exam).

- `date`: When it’s due (day count from start).

- `weight`: How much it counts toward the final grade (usually all assignments = 100%, and the Exam = a separate 100%).

In [ ]:
print(f"Shape: {df_assessments.shape[0]} rows × {df_assessments.shape[1]} columns\n")
print("Columns:")
print(df_assessments.dtypes)
print("\nFirst 5 rows:")
print(df_assessments.head())
print("\nMissing values:")
print(df_assessments.isnull().sum())
print("\nAssessment types:")
print(df_assessments['assessment_type'].value_counts())
print(f"\nTotal weight: {df_assessments['weight'].sum()}")
print("\nDate range:")
print(f"Min date: {df_assessments['date'].min()}")
print(f"Max date: {df_assessments['date'].max()}")
print("\nAssessments per module:")
print(df_assessments['code_module'].value_counts())

Shape: 206 rows × 6 columns

Columns:
code_module           object
code_presentation     object
id_assessment          int64
assessment_type       object
date                 float64
weight               float64
dtype: object

First 5 rows:
  code_module code_presentation  id_assessment assessment_type   date  weight
0         AAA             2013J           1752             TMA   19.0    10.0
1         AAA             2013J           1753             TMA   54.0    20.0
2         AAA             2013J           1754             TMA  117.0    20.0
3         AAA             2013J           1755             TMA  166.0    20.0
4         AAA             2013J           1756             TMA  215.0    30.0

Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Assessment types:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

Total weight: 4300.

<span style = "font-family: Verdana; font-size: 15px">

### studentAssessment.csv
Results of students' assessments. If the student does not submit the assessment, no result is recorded. The final exam submissions is missing, if the result of the assessments is not stored in the system.

- `id_assessment` - **foreign key** from *assessment.csv*.

- `id_student` - **foreign key** from *studentInfo.csv*.

- `date_submitted`: the date of student submission, measured as the number of days since the start of the module presentation.

- `is_banked`: a status flag indicating that the assessment result has been transferred from a previous presentation.

- `score`: the student's score in this assessment. The range is from 0 to 100. The score lower than 40 is interpreted as Fail. The marks are in the range from 0 to 100.

In [13]:
print(f"Shape: {df_student_assessment.shape[0]} rows × {df_student_assessment.shape[1]} columns\n")
print("Columns:")
print(df_student_assessment.dtypes)
print("\nFirst 5 rows:")
print(df_student_assessment.head())
print("\nMissing values:")
print(df_student_assessment.isnull().sum())
print("\nScore summary:")
print(df_student_assessment['score'].describe())
print(f"\nStudents in assessment data: {df_student_assessment['id_student'].nunique()}")
print(f"Total assessment submissions: {len(df_student_assessment)}")
print("\nDate range:")
print(f"Min date: {df_student_assessment['date_submitted'].min()}")
print(f"Max date: {df_student_assessment['date_submitted'].max()}")

Shape: 173912 rows × 5 columns

Columns:
id_assessment       int64
id_student          int64
date_submitted      int64
is_banked           int64
score             float64
dtype: object

First 5 rows:
   id_assessment  id_student  date_submitted  is_banked  score
0           1752       11391              18          0   78.0
1           1752       28400              22          0   70.0
2           1752       31604              17          0   72.0
3           1752       32885              26          0   69.0
4           1752       38053              19          0   79.0

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

Score summary:
count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64

Students in assessment data: 23369
Total assessment s

<span style = "font-family: Verdana; font-size: 15px">

### studentInfo.csv
Demographic information about the students together with their results.

- `code_module` - **foreign key** from *courses.csv*.

- `code_presentation` - **foreign key** from *courses.csv*.

- `id_student`: a unique identification number for the student. - **primary key**

- `gender`: the student's gender.

- `region`: identifies the geographic region, where the student lived while taking the module-presentation.

- `highest_education`: highest student education level on entry to the module presentation.

- `imd_band`: specifies the Index of Multiple Depravation band of the place where the student lived during the module-presentation.

- `age_band`: band of the student's age.

- `num_of_prev_attempts`: the number times the student has attempted this module.

- `studied_credits`: the total number of credits for the modules the student is currently studying.

- `disability`: indicates whether the student has declared a disability.

- `final_result`: student's final result in the module-presentation.

In [4]:
print(f"Shape: {df_student_info.shape[0]} rows × {df_student_info.shape[1]} columns\n")
print("Columns:")
print(df_student_info.dtypes)
print("\nFirst 5 rows:")
print(df_student_info.head())
print("\nMissing values:")
print(df_student_info.isnull().sum())
print("\nSummary statistics:")
print(df_student_info.describe())
print("\nGender distribution:")
print(df_student_info['gender'].value_counts())
print("\nAge band distribution:")
print(df_student_info['age_band'].value_counts().sort_index())
print("\nFinal result distribution:")
print(df_student_info['final_result'].value_counts())

Shape: 32593 rows × 12 columns

Columns:
code_module             object
code_presentation       object
id_student               int64
gender                  object
region                  object
highest_education       object
imd_band                object
age_band                object
num_of_prev_attempts     int64
studied_credits          int64
disability              object
final_result            object
dtype: object

First 5 rows:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604      F     South East Region   
4         AAA             2013J       32885      F  West Midlands Region   

       highest_education imd_band age_band  num_of_prev_attempts  \
0       HE Qualification  90-100%

<span style = "font-family: Verdana; font-size: 15px">

### studentRegistration.csv
Information about the time when the student registered for the module presentation. For students who unregistered the unregistered date is also recorded.

- `code_module` - **foreign key** from *courses.csv*.
- `code_presentation` - **foreign key** from *courses.csv*.
- `id_student` - **foreign key** from *studentInfo.csv*.
- `date_registration`: the date of student's registration on the module presentation, this is the number of days measured relative to the start of the module-presentation (e.g. the negative value -30 means that the student registered to module presentation 30 days before it started).
- `date_unregistration`: the student's unregistered date from the module presentation, this is the number of days measured relative to the start of the module-presentation. Students, who completed the course have this field empty. Students who unregistered have Withdrawal as the value of the final_result column in the studentInfo.csv file.

In [5]:
print(f"Shape: {df_student_registration.shape[0]} rows × {df_student_registration.shape[1]} columns\n")
print("Columns:")
print(df_student_registration.dtypes)
print("\nFirst 5 rows:")
print(df_student_registration.head())
print("\nMissing values:")
print(df_student_registration.isnull().sum())
print("\nRegistration dates:")
print(f"Date range: {df_student_registration['date_registration'].min()} to {df_student_registration['date_registration'].max()}")
print("\nStudents per module:")
print(df_student_registration['code_module'].value_counts())
print("\nCourse presentation distribution:")
print(df_student_registration['code_presentation'].value_counts().sort_index())

Shape: 32593 rows × 5 columns

Columns:
code_module             object
code_presentation       object
id_student               int64
date_registration      float64
date_unregistration    float64
dtype: object

First 5 rows:
  code_module code_presentation  id_student  date_registration  \
0         AAA             2013J       11391             -159.0   
1         AAA             2013J       28400              -53.0   
2         AAA             2013J       30268              -92.0   
3         AAA             2013J       31604              -52.0   
4         AAA             2013J       32885             -176.0   

   date_unregistration  
0                  NaN  
1                  NaN  
2                 12.0  
3                  NaN  
4                  NaN  

Missing values:
code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

Registration dates:
Date range: -322.0 to 167.0

Stud

<span style = "font-family: Verdana; font-size: 15px">

### vle.csv

Information about the available materials in the VLE

- `id_site`: an identification number of the material - **primary key**

- `code_module` - **foreign key** from *courses.csv*

- `code_presentation` - **foreign key** from *courses.csv*

- `activity_type`: the role associated with the module material

- `week_from`: the week from which the material is planned to be used

- `week_to`: week until which the material is planned to be used

In [6]:
print(f"Shape: {df_vle.shape[0]} rows × {df_vle.shape[1]} columns\n")
print("Columns:")
print(df_vle.dtypes)
print("\nFirst 5 rows:")
print(df_vle.head())
print("\nMissing values:")
print(df_vle.isnull().sum())
print("\nActivity types:")
print(df_vle['activity_type'].value_counts())
print(f"\nTotal unique activities: {df_vle['id_site'].nunique()}")
print(f"Week range: {df_vle['week_from'].min()} to {df_vle['week_to'].max()} weeks")

Shape: 6364 rows × 6 columns

Columns:
id_site                int64
code_module           object
code_presentation     object
activity_type         object
week_from            float64
week_to              float64
dtype: object

First 5 rows:
   id_site code_module code_presentation activity_type  week_from  week_to
0   546943         AAA             2013J      resource        NaN      NaN
1   546712         AAA             2013J     oucontent        NaN      NaN
2   546998         AAA             2013J      resource        NaN      NaN
3   546888         AAA             2013J           url        NaN      NaN
4   547035         AAA             2013J      resource        NaN      NaN

Missing values:
id_site                 0
code_module             0
code_presentation       0
activity_type           0
week_from            5243
week_to              5243
dtype: int64

Activity types:
activity_type
resource          2660
subpage           1055
oucontent          996
url                886

<span style = "font-family: Verdana; font-size: 15px">

### studentVle.csv
Information about each student's interactions with the materials in the VLE.

- `code_module` - **foreign key** from *courses.csv*.

- `code_presentation` - **foreign key** from *courses.csv*.

- `id_student` - **foreign key** from *studentInfo.csv*.

- `id_site` - **foreign key** from *vle.csv*.

- `date`: the date of student's interaction with the material measured as the number of days since the start of the module-presentation.

- `sum_click`: the number of times a student interacts with the material in that day.

In [11]:
print(f"Shape: {df_student_vle.shape[0]} rows × {df_student_vle.shape[1]} columns\n")
print("Columns:")
print(df_student_vle.dtypes)
print("\nFirst 5 rows:")
print(df_student_vle.head())
print("\nMissing values:")
print(df_student_vle.isnull().sum())
print("\nSummary statistics (clicks):")
print(df_student_vle['sum_click'].describe())
print(f"\nDate range: {df_student_vle['date'].min()} to {df_student_vle['date'].max()}")
print(f"Students in VLE data: {df_student_vle['id_student'].nunique()}")
print(f"Unique activities: {df_student_vle['id_site'].nunique()}")
print("\nTop 10 activities by total clicks:")
print(df_student_vle.groupby('id_site')['sum_click'].sum().sort_values(ascending=False).head(10))

Shape: 10655280 rows × 6 columns

Columns:
code_module          object
code_presentation    object
id_student            int64
id_site               int64
date                  int64
sum_click             int64
dtype: object

First 5 rows:
  code_module code_presentation  id_student  id_site  date  sum_click
0         AAA             2013J       28400   546652   -10          4
1         AAA             2013J       28400   546652   -10          1
2         AAA             2013J       28400   546652   -10          1
3         AAA             2013J       28400   546614   -10         11
4         AAA             2013J       28400   546714   -10          1

Missing values:
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

Summary statistics (clicks):
count    1.065528e+07
mean     3.716946e+00
std      8.849047e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3

<span style = "font-family: Verdana; font-size: 15px">

### Key Insights

In [14]:
at_risk_count = (df_student_info['final_result'] == 'Withdrawn').sum()
print(f"Students withdrawn (at-risk): {at_risk_count} ({at_risk_count/len(df_student_info)*100:.1f}%)")

fail_count = (df_student_info['final_result'] == 'Fail').sum()
print(f"Students failed: {fail_count} ({fail_count/len(df_student_info)*100:.1f}%)")

pass_count = (df_student_info['final_result'] == 'Pass').sum()
print(f"Students passed: {pass_count} ({pass_count/len(df_student_info)*100:.1f}%)")

Students withdrawn (at-risk): 10156 (31.2%)
Students failed: 7052 (21.6%)
Students passed: 12361 (37.9%)


<span style = "font-family: Verdana; font-size: 15px">

### Data Quality Summary

In [15]:
print(f"Total students: {df_student_info['id_student'].nunique()}")
print(f"Total modules: {df_courses['code_module'].nunique()}")
print(f"VLE activity records: {len(df_student_vle):,}")
print(f"Assessment records: {len(df_student_assessment):,}")
print(f"Student registrations: {len(df_student_registration):,}")

Total students: 28785
Total modules: 7
VLE activity records: 10,655,280
Assessment records: 173,912
Student registrations: 32,593
